# 03 - Inference
Three equivalent ways to score a patient: the packaged joblib pipeline, the MLflow model (registry format) and the REST API.

In [1]:
import json, warnings
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown
warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
from heartrisk import config
from heartrisk.predict import ModelService
svc = ModelService()
print("model:", svc.version, "| type:", svc.model_type, "| threshold:", svc.threshold)

model: heart-disease-classifier/v1 | type: logistic_regression | threshold: 0.31


In [2]:
patients = [json.loads((config.PROJECT_ROOT / "samples" / f).read_text())
            for f in ("patient_high_risk.json", "patient_low_risk.json")]
pd.DataFrame([p.to_dict() for p in svc.predict(patients)], index=["high-risk sample", "low-risk sample"])

,prediction,label,probability,confidence,risk_band,threshold
high-risk sample,1,heart disease,0.9773,0.9773,high,0.31
low-risk sample,0,no heart disease,0.1148,0.8852,low,0.31


### Missing optional fields are imputed by the pipeline

In [3]:
partial = dict(patients[0]); partial.pop("ca"); partial.pop("thal")
svc.predict(partial)[0].to_dict()

{'prediction': 1,
 'label': 'heart disease',
 'probability': 0.7976,
 'confidence': 0.7976,
 'risk_band': 'high',
 'threshold': 0.31}

### Same model through the MLflow pyfunc interface

In [4]:
import mlflow
m = mlflow.pyfunc.load_model(str(config.MODELS_DIR / "mlflow_model"))
from heartrisk.features import to_frame
print(m.metadata.get_input_schema())
m.predict(to_frame(patients).astype(float))

['age': double (required), 'trestbps': double (required), 'chol': double (required), 'thalach': double (required), 'oldpeak': double (required), 'ca': double (required), 'sex': double (required), 'fbs': double (required), 'exang': double (required), 'cp': double (required), 'restecg': double (required), 'slope': double (required), 'thal': double (required)]


array([[0.02274088, 0.97725912],
       [0.88516481, 0.11483519]])

### Batch scoring of the full dataset

In [5]:
from heartrisk.data import load_clean
df = load_clean()
proba = svc.predict_proba(df[config.FEATURES])
df.assign(probability=proba.round(3), predicted=(proba >= svc.threshold).astype(int))[
    ["age", "sex", "cp", "thalach", "oldpeak", "ca", "thal", "target", "probability", "predicted"]].head(10)

,age,sex,cp,thalach,oldpeak,ca,thal,target,probability,predicted
0,63.0,1,1,150.0,2.3,0,6,0,0.429,1
1,67.0,1,4,108.0,1.5,3,3,1,0.977,1
2,67.0,1,4,129.0,2.6,2,7,1,0.969,1
3,37.0,1,3,187.0,3.5,0,3,0,0.185,0
4,41.0,0,2,172.0,1.4,0,3,0,0.115,0
5,56.0,1,2,178.0,0.8,0,3,0,0.113,0
6,62.0,0,4,160.0,3.6,2,3,1,0.766,1
7,57.0,0,4,163.0,0.6,0,3,0,0.225,0
8,63.0,1,4,147.0,1.4,1,7,1,0.872,1
9,53.0,1,4,155.0,3.1,0,7,1,0.800,1


### REST API (if running: `make serve` or `docker run -p 8000:8000 heart-risk-api:1.0.0`)

In [6]:
import requests
try:
    r = requests.post("http://localhost:8000/predict", json=patients[0], timeout=3)
    print(r.status_code, r.headers.get("X-Request-ID")); print(json.dumps(r.json(), indent=2))
except requests.ConnectionError:
    print("API not running locally - start it with `make serve` to try this cell.")

200 03b4381275d64672
{
  "prediction": 1,
  "label": "heart disease",
  "probability": 0.9773,
  "confidence": 0.9773,
  "risk_band": "high",
  "threshold": 0.31,
  "model_version": "heart-disease-classifier/v1",
  "request_id": "03b4381275d64672"
}
